In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import optuna
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

# ── Hardware Optimization ────────────────────────────────────────────────────
torch.set_num_threads(2) 
device0 = torch.device("cuda:0")
gpu_count = torch.cuda.device_count()
scaler_amp = GradScaler(device='cuda')

print(f"🚀 UVA HPC Environment Active | GPUs: {gpu_count}x RTX 2080 (11GB)")

/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 UVA HPC Environment Active | GPUs: 2x RTX 2080 (11GB)


In [2]:
# ── Data Loading & Split Logic ───────────────────────────────────────────────
FILE_PATH = "delineated_storms.parquet"
df = pd.read_parquet(FILE_PATH)

# Features & Metadata
features = ['precip_1hr [inch]', 'precip_max_intensity [inch/hour]', 
            'temp_2m [degF]', 'soil_moisture_05cm [m^3/m^3]', 'elevation [feet]']
target   = 'depth_inches'

# Dictionary-based cleaning to avoid Ambiguous Truth Value errors
df[features + [target]] = df[features + [target]].replace({np.inf: np.nan, -np.inf: np.nan})
df = df.dropna(subset=features + [target])

# Temporal Split by Storm ID
storm_info = df.groupby('global_storm_id')['time'].min().sort_values()
storm_ids  = storm_info.index.tolist()

train_idx = int(len(storm_ids) * 0.70)
val_idx   = int(len(storm_ids) * 0.85)

train_ids, val_ids, test_ids = storm_ids[:train_idx], storm_ids[train_idx:val_idx], storm_ids[val_idx:]

train_df = df[df['global_storm_id'].isin(train_ids)].sort_values(['global_storm_id', 'time'])
val_df   = df[df['global_storm_id'].isin(val_ids)].sort_values(['global_storm_id', 'time'])
test_df  = df[df['global_storm_id'].isin(test_ids)].sort_values(['global_storm_id', 'time'])

print(f"📊 Split Totals: {len(train_ids)} Train | {len(val_ids)} Val | {len(test_ids)} Test Storms")

📊 Split Totals: 6566 Train | 1407 Val | 1407 Test Storms


In [3]:
# ── Scaling ──────────────────────────────────────────────────────────────────
scaler_X, scaler_y = StandardScaler(), StandardScaler()

X_train_s = scaler_X.fit_transform(train_df[features].values).astype('float32')
y_train_s = scaler_y.fit_transform(train_df[[target]].values).astype('float32')
X_val_s   = scaler_X.transform(val_df[features].values).astype('float32')
X_test_s  = scaler_X.transform(test_df[features].values).astype('float32')

# GPU residency for non-sequential models
X_train_gpu = torch.tensor(X_train_s, device=device0)
y_train_gpu = torch.tensor(y_train_s, device=device0)
X_val_gpu   = torch.tensor(X_val_s, device=device0)
y_val_raw   = torch.tensor(val_df[target].values, device=device0, dtype=torch.float32)

del df; gc.collect()

66

In [4]:
# ── Model & Dataset Definitions ──────────────────────────────────────────────
class SotaANN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, x): return self.net(x)

class SotaAttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, 2, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.fc   = nn.Linear(hidden_size * 2, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        w = F.softmax(self.attn(out), dim=1)
        return self.fc(torch.sum(out * w, dim=1))

class StormWindowDataset(Dataset):
    def __init__(self, df_split, window_size):
        self.window, self.segments, self.index = window_size, [], []
        for _, grp in df_split.groupby('global_storm_id', sort=False):
            if len(grp) <= window_size: continue
            self.segments.append((scaler_X.transform(grp[features].values), grp[target].values))
            for start in range(len(grp) - window_size):
                self.index.append((len(self.segments)-1, start))
    def __len__(self): return len(self.index)
    def __getitem__(self, idx):
        seg, start = self.index[idx]
        X_s, y_s = self.segments[seg]
        return torch.tensor(X_s[start:start+self.window], dtype=torch.float32), torch.tensor(y_s[start+self.window], dtype=torch.float32)

def nse_metric(y_true, y_pred):
    denom = torch.sum((y_true - torch.mean(y_true)) ** 2)
    return 1 - (torch.sum((y_true - y_pred) ** 2) / (denom + 1e-8))

In [5]:
# ── Optuna Objectives ────────────────────────────────────────────────────────
_cache = {}
def get_cached_ds(split, win):
    if (split, win) not in _cache:
        df_obj = {'train': train_df, 'val': val_df}[split]
        _cache[(split, win)] = StormWindowDataset(df_obj, win)
    return _cache[(split, win)]

def objective_lr(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    c = trial.suggest_float("log_shift", 1e-4, 0.5, log=True)
    y_tr_log = np.log(np.maximum(train_df[target].values + c, 1e-9))
    model = Ridge(alpha=alpha).fit(X_train_s, y_tr_log)
    preds = np.exp(model.predict(X_val_s)) - c
    y_val = val_df[target].values
    return 1 - (np.sum((y_val - preds)**2) / (np.sum((y_val - np.mean(y_val))**2) + 1e-8))

def objective_ann(trial):
    h = trial.suggest_int("hidden_size", 128, 512)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    model = SotaANN(len(features), h).to(device0)
    if gpu_count > 1: model = nn.DataParallel(model)
    opt = optim.Adam(model.parameters(), lr=lr)
    for _ in range(5):
        for i in range(0, len(X_train_gpu), 32768):
            bx, by = X_train_gpu[i:i+32768], y_train_gpu[i:i+32768]
            opt.zero_grad(); loss = nn.MSELoss()(model(bx), by)
            scaler_amp.scale(loss).backward(); scaler_amp.step(opt); scaler_amp.update()
    model.eval()
    with torch.no_grad():
        p = model(X_val_gpu).cpu().flatten() * scaler_y.scale_[0] + scaler_y.mean_[0]
        return nse_metric(y_val_raw.cpu(), p).item()

def objective_lstm(trial):
    win = trial.suggest_int("window_size", 30, 90, step=30)
    h = trial.suggest_int("hidden_size", 32, 128)
    loader = DataLoader(get_cached_ds('train', win), batch_size=2048, shuffle=False)
    model = SotaAttentionLSTM(len(features), h).to(device0)
    if gpu_count > 1: model = nn.DataParallel(model)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(2):
        for bx, by in loader:
            bx, by_s = bx.to(device0), ((by - scaler_y.mean_[0])/scaler_y.scale_[0]).to(device0).unsqueeze(1)
            opt.zero_grad(); loss = nn.MSELoss()(model(bx), by_s)
            scaler_amp.scale(loss).backward(); scaler_amp.step(opt); scaler_amp.update()
    model.eval()
    v_loader = DataLoader(get_cached_ds('val', win), batch_size=2048)
    preds, trues = [], []
    with torch.no_grad():
        for bx, by in v_loader:
            p = model(bx.to(device0)).cpu().flatten() * scaler_y.scale_[0] + scaler_y.mean_[0]
            preds.append(p); trues.append(by)
    return nse_metric(torch.cat(trues), torch.cat(preds)).item()

In [6]:
print("🏁 Starting Multi-Model Shootout...")
study_lr = optuna.create_study(direction="maximize"); study_lr.optimize(objective_lr, n_trials=30)
study_ann = optuna.create_study(direction="maximize"); study_ann.optimize(objective_ann, n_trials=20)
study_lstm = optuna.create_study(direction="maximize"); study_lstm.optimize(objective_lstm, n_trials=10)

print(f"🏆 Best NSEs | LR: {study_lr.best_value:.4f} | ANN: {study_ann.best_value:.4f} | LSTM: {study_lstm.best_value:.4f}")

[I 2026-04-08 14:15:07,246] A new study created in memory with name: no-name-3c85aa93-8721-4137-a7b3-325d49fcec37
[I 2026-04-08 14:15:07,386] Trial 0 finished with value: -0.006768051225888527 and parameters: {'alpha': 0.0049527728128202055, 'log_shift': 0.020761331357669468}. Best is trial 0 with value: -0.006768051225888527.


🏁 Starting Multi-Model Shootout...


[I 2026-04-08 14:15:07,514] Trial 1 finished with value: -0.007968311943253203 and parameters: {'alpha': 0.0231540915143536, 'log_shift': 0.00011801055462074076}. Best is trial 0 with value: -0.006768051225888527.
[I 2026-04-08 14:15:07,641] Trial 2 finished with value: -0.007706097925385791 and parameters: {'alpha': 0.012199588393076291, 'log_shift': 0.0031007101499010868}. Best is trial 0 with value: -0.006768051225888527.
[I 2026-04-08 14:15:07,768] Trial 3 finished with value: -0.00129643459150941 and parameters: {'alpha': 0.01140988560505708, 'log_shift': 0.30662589583596916}. Best is trial 3 with value: -0.00129643459150941.
[I 2026-04-08 14:15:07,895] Trial 4 finished with value: -0.007937366746758823 and parameters: {'alpha': 5.2616951156614595, 'log_shift': 0.00037646962891421166}. Best is trial 3 with value: -0.00129643459150941.
[I 2026-04-08 14:15:08,021] Trial 5 finished with value: -0.007942729333223486 and parameters: {'alpha': 0.007113789777862653, 'log_shift': 0.000328

OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/torch/nn/parallel/parallel_apply.py", line 114, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_40802/4123293634.py", line 20, in forward
    out, _ = self.lstm(x)
             ^^^^^^^^^^^^
  File "/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_

In [ ]:
# 1. Best Log-Reg
b_lr = study_lr.best_params
lr_final = Ridge(alpha=b_lr['alpha']).fit(X_train_s, np.log(np.maximum(train_df[target].values + b_lr['log_shift'], 1e-9)))
lr_preds = np.exp(lr_final.predict(X_test_s)) - b_lr['log_shift']

# 2. Best ANN
b_ann = study_ann.best_params
ann_final = SotaANN(len(features), b_ann['hidden_size']).to(device0)
if gpu_count > 1: ann_final = nn.DataParallel(ann_final)
opt_ann = optim.Adam(ann_final.parameters(), lr=b_ann['lr'])
for _ in range(15):
    for i in range(0, len(X_train_gpu), 32768):
        bx, by = X_train_gpu[i:i+32768], y_train_gpu[i:i+32768]
        opt_ann.zero_grad(); loss = nn.MSELoss()(ann_final(bx), by)
        scaler_amp.scale(loss).backward(); scaler_amp.step(opt_ann); scaler_amp.update()

# 3. Best LSTM
b_lstm = study_lstm.best_params
lstm_final = SotaAttentionLSTM(len(features), b_lstm['hidden_size']).to(device0)
if gpu_count > 1: lstm_final = nn.DataParallel(lstm_final)
opt_lstm = optim.Adam(lstm_final.parameters(), lr=1e-3)
f_loader = DataLoader(StormWindowDataset(train_df, b_lstm['window_size']), batch_size=2048)
for _ in range(5):
    for bx, by in f_loader:
        bx, by_s = bx.to(device0), ((by - scaler_y.mean_[0])/scaler_y.scale_[0]).to(device0).unsqueeze(1)
        opt_lstm.zero_grad(); loss = nn.MSELoss()(lstm_final(bx), by_s)
        scaler_amp.scale(loss).backward(); scaler_amp.step(opt_lstm); scaler_amp.update()

In [ ]:
def plot_final_research_report():
    ann_final.eval(); lstm_final.eval()
    with torch.no_grad():
        X_test_gpu = torch.tensor(X_test_s, device=device0)
        ann_test_p = (ann_final(X_test_gpu).cpu().flatten() * scaler_y.scale_[0] + scaler_y.mean_[0]).numpy()
        
        test_loader = DataLoader(StormWindowDataset(test_df, b_lstm['window_size']), batch_size=2048)
        lp, lt = [], []
        for bx, by in test_loader:
            p = lstm_final(bx.to(device0)).cpu().flatten() * scaler_y.scale_[0] + scaler_y.mean_[0]
            lp.append(p); lt.append(by)
        lstm_p, lstm_t = torch.cat(lp).numpy(), torch.cat(lt).numpy()

    fig = plt.figure(figsize=(18, 10), dpi=150)
    gs = fig.add_gridspec(2, 2)
    
    # Hydrograph (Temporal Peak Analysis)
    ax1 = fig.add_subplot(gs[0, :])
    win = slice(-1000, -200) # Event window
    ax1.plot(test_df[target].values[win], label='Observed (FloodNet)', color='black', lw=3, alpha=0.8)
    ax1.plot(lr_preds[win], label='Log-Ridge Baseline', color='#e67e22', ls='--')
    ax1.plot(ann_test_p[win], label='SOTA Res-ANN', color='#9b59b6')
    ax1.plot(lstm_p[win], label='Attn-LSTM (Windowed)', color='#2ecc71', lw=2)
    ax1.set_title("Temporal Flood Accuracy: NYC Catchment Peak Analysis", fontsize=16, fontweight='bold')
    ax1.set_ylabel("Depth (in)"); ax1.legend(); ax1.grid(True, alpha=0.3)
    
    # ANN Scatter
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.scatter(test_df[target].values, ann_test_p, alpha=0.1, color='#9b59b6', s=5)
    ax2.plot([0, 15], [0, 15], 'k--'); ax2.set_title("ANN: Residual Bias Analysis")
    
    # LSTM Scatter
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.scatter(lstm_t, lstm_p, alpha=0.1, color='#2ecc71', s=5)
    ax3.plot([0, 15], [0, 15], 'k--'); ax3.set_title("LSTM: Sequential Bias Analysis")
    
    plt.tight_layout(); plt.savefig('UVA_MSDS_Flood_Report.png'); plt.show()
    torch.cuda.empty_cache(); gc.collect()

plot_final_research_report()